## byLLM Script

Description here

In [1]:
import os
import pandas as pd
from byllm.lib import Model, by

# This pulls the value from the system variable
apiKey = os.getenv('LLM_API_KEY')

if not apiKey:
    raise ValueError("Missing API Key! Please set the LLM_API_KEY environment variable.")

llm = Model(model_name="gemini/gemini-2.5-flash" , api_key= apiKey)

## Import Datasets

In [9]:
filePath = '../../data/fdic/fdic_250.csv'
df = pd.read_csv(filePath)

#df.head()

In [3]:
# Create df2 for a smaller sample in testing
df2 = df.iloc[:25, :5]
#df2.shape

## byLLM model integration

In [4]:
import pandas as pd
import time
from dataclasses import dataclass, asdict

@dataclass
class TextBatch:
    values: list[str]

@by(llm)
def clean_column_values(batch: TextBatch) -> TextBatch:
    """
    Clean this list of strings
    """
    ...

def clean_entire_dataframe(df: pd.DataFrame):
    cleaned_df = df.copy().astype(str)
    
    for col in df.columns:
        print(f"--- Cleaning Column: {col} ---")
        column_data = cleaned_df[col].tolist()
        new_column_values = []
        
        # Process the column in chunks of 50 values (much faster!)
        batch_size = 50 
        for i in range(0, len(column_data), batch_size):
            chunk = column_data[i : i + batch_size]
            
            try:
                # Wrap in dataclass for byllm
                result = clean_column_values(TextBatch(values=chunk))
                new_column_values.extend(result.values)
                
                # Small sleep to respect the 15 RPM limit
                time.sleep(4) 
                
            except Exception as e:
                print(f"Error in {col} at index {i}: {e}")
                new_column_values.extend(chunk) # Keep original on failure
                time.sleep(20)
        
        # Update the column in our dataframe
        # Ensure the lengths match before assignment
        if len(new_column_values) == len(cleaned_df):
            cleaned_df[col] = new_column_values
            
    return cleaned_df

# --- Execution ---
# df_fully_cleaned = clean_entire_dataframe(df2)

## Slice the first 5 columns for rate limitting & Run Model

Current model should be run with 5 columns x 25 rows

In [5]:
clean_data = clean_entire_dataframe(df2)

--- Cleaning Column: fdic_certificate_number ---
--- Cleaning Column: institution_name ---
--- Cleaning Column: state_name ---
--- Cleaning Column: fdic_id ---
--- Cleaning Column: docket ---


In [6]:
#Expected (25, 5)
clean_data.shape

(25, 5)

In [8]:
clean_data.to_csv('../../data/fdic/byllm_cleaned_fdic.csv', index=False)